In [18]:
import pandas as pd

In [19]:
df = pd.read_parquet("../data/candidate_detail.parquet")

In [20]:
pd.set_option("display.max_rows", None)

In [21]:
c = pd.read_parquet("../data/candidate_detail.parquet")
c = c[c.outcome.isin(["Approved","Commercialized","Failed Phase 1","Failed Phase 2","Failed Phase 3"])].copy()
c["y"]     = c.outcome.isin(["Approved","Commercialized"]).astype(int)
c["drug"]  = c.candidate_id.str.split("__").str[0]          # drug identity (memorization key)
c["year"]  = pd.to_datetime(c.earliest_start_date, errors="coerce").dt.year
c["split"] = (c.year <= 2019).map({True:"train", False:"test"})

In [22]:
# join the model's test predictions to inspect seen/new-drug behavior:
pred = pd.read_parquet("../runs/xgb_di_md/predictions.parquet")  # candidate_id, y_true, y_proba
df = pred.merge(c[["candidate_id","drug","outcome","year"]], right_on="candidate_id", left_on="example_id", how="left")
df["seen_drug"] = df.drug.isin(set(c[c.split=="train"].drug))

In [23]:
df.seen_drug.value_counts()

seen_drug
True     2378
False     259
Name: count, dtype: int64

In [ ]:
groups = df[df.year > 2019].groupby("drug")
for drug, group in groups:
    if len(group) == 1: 
        continue
    if group.label.sum() == len(group) or group.label.sum() == 0:
        continue

    weakest_approval = group[group.label==1].y_proba.min()
    strongest_failure = group[group.label==0].y_proba.max()
    if weakest_approval < strongest_failure:
        continue
    print(f"Drug: {drug}")
    print(group[["example_id","outcome","label","y_proba","seen_drug"]])
    print("-" * 40)


Drug: db:DB00007
                                     example_id         outcome  label  \
9                   db:DB00007__obesity, morbid        Approved      1   
595                         db:DB00007__bulimia  Failed Phase 2      0   
1776  db:DB00007__primary ovarian insufficiency  Failed Phase 1      0   

       y_proba  seen_drug  
9     0.066498       True  
595   0.131747       True  
1776  0.189642       True  
----------------------------------------
Drug: db:DB00107
                                        example_id         outcome  label  \
2                      db:DB00107__placenta previa  Failed Phase 3      0   
126   db:DB00107__adverse effect of oxytocic drugs  Failed Phase 3      0   
1270             db:DB00107__diabetes, gestational        Approved      1   
1337            db:DB00107__coronavirus infections  Failed Phase 2      0   
1482                         db:DB00107__leiomyoma  Failed Phase 1      0   
1713               db:DB00107__preeclampsia severe    

In [13]:
df

,example_id,label,phase,y_proba,candidate_id,drug,outcome,year,seen_drug
0,"db:DB08880__multiple sclerosis, chronic progre...",1,P3,0.646630,"db:DB08880__multiple sclerosis, chronic progre...",db:DB08880,Approved,2024,True
1,"db:DB00316__lymphoma, large b-cell, diffuse",0,P2,0.365333,"db:DB00316__lymphoma, large b-cell, diffuse",db:DB00316,Failed Phase 2,2021,True
2,db:DB00107__placenta previa,0,P4,0.490918,db:DB00107__placenta previa,db:DB00107,Failed Phase 3,2022,True
3,"db:DB00582__leishmaniasis, cutaneous",0,P3,0.402385,"db:DB00582__leishmaniasis, cutaneous",db:DB00582,Failed Phase 3,2022,True
4,name:pf 06650833__covid-19,0,P2,0.424751,name:pf 06650833__covid-19,name:pf 06650833,Failed Phase 2,2020,True
5,db:DB01156__hepatolenticular degeneration,0,P1,0.503689,db:DB01156__hepatolenticular degeneration,db:DB01156,Failed Phase 1,2020,True
6,db:DB06292__kidney diseases,1,P3,0.559593,db:DB06292__kidney diseases,db:DB06292,Approved,2021,True
7,db:DB00235__anesthesia,0,P4,0.325242,db:DB00235__anesthesia,db:DB00235,Failed Phase 3,2021,True
8,"name:a__diabetes mellitus, type 2",0,P1,0.368281,"name:a__diabetes mellitus, type 2",name:a,Failed Phase 1,2023,True
9,"db:DB00007__obesity, morbid",1,P2,0.094088,"db:DB00007__obesity, morbid",db:DB00007,Approved,2020,True


In [11]:
df[df.example_id.str.contains("exenatide")]

,example_id,label,phase,y_proba,candidate_id,drug,outcome,year,seen_drug


In [7]:
# ROC AUC on seen vs. new drugs:
from sklearn.metrics import roc_auc_score
print("ROC AUC on seen drugs:", roc_auc_score(df[df.seen_drug].label, df[df.seen_drug].y_proba))
print("ROC AUC on new drugs:", roc_auc_score(df[~df.seen_drug].label, df[~df.seen_drug].y_proba))

ROC AUC on seen drugs: 0.5948137223209956
ROC AUC on new drugs: 0.5760854953622798


In [8]:
# PR AUC on seen vs. new drugs:
from sklearn.metrics import average_precision_score
print("PR AUC on seen drugs:", average_precision_score(df[df.seen_drug].label, df[df.seen_drug].y_proba))
print("PR AUC on new drugs:", average_precision_score(df[~df.seen_drug].label, df[~df.seen_drug].y_proba))

PR AUC on seen drugs: 0.3212923759901194
PR AUC on new drugs: 0.3962363113148435
